# Module 2 -- Multi-Provider Routing and Fallbacks

## Before You Start

Make sure your Module 1 Docker stack is still running:

```powershell
cd tensorzero-demo
docker compose ps
```

You should see `gateway`, `postgres`, and `ui` all in `Up (healthy)` state.  
If not, run `docker compose up -d --wait` to start it again.

In this module we will **only edit `tensorzero.toml`** and restart the gateway.  
No changes to `docker-compose.yml` or `.env`.

---

---
# Part 1 -- Theory: Why Multi-Provider Routing?

---

## 1.1 -- The Production Reality

Your app works perfectly in development. Then you ship to production and:

```
Monday 2am:  OpenAI has an outage -- all your requests return 503
Tuesday:     You hit OpenAI's rate limit -- requests queued for minutes
Wednesday:   OpenAI raises prices 20% -- your cost budget breaks
Thursday:    A new Groq model is 3x faster and half the price
Friday:      Your on-call gets paged because one model degraded in quality
```

Every single one of these is a **provider-level problem**, not a code problem.  
Without a gateway, fixing each one requires code changes + redeploy.

### The Solution: Multi-Provider Routing

```
Request arrives
      |
      v
TensorZero Gateway
      |
      +---> Try OpenAI first     (200 OK? done.)
      |
      +---> OpenAI failed?       (503, 429, timeout?)
      |     Try Groq instead     (200 OK? done. user never knew.)
      |
      +---> Groq also failed?    (rare, but possible)
            Return error to user (after exhausting all providers)
```

This is **transparent failover** -- your application code never changes.

## 1.2 -- TensorZero's Routing Architecture

TensorZero has **two levels** where routing decisions happen:

### Level 1: Provider Routing (within a Model)

```
Model: "resilient_gpt"
  routing = ["openai", "groq"]    <-- ordered list of providers
  |
  +-- Provider "openai"           <-- try first
  |     type = openai
  |     model_name = gpt-4o-mini
  |
  +-- Provider "groq"             <-- fallback if openai fails
        type = groq
        model_name = llama-3.1-8b-instant
```

This is **infrastructure-level resilience** -- same logical model, different hosts.  
Use this when: provider outage, rate limits, latency spikes.

### Level 2: Variant Routing (within a Function)

```
Function: "chat"
  |
  +-- Variant "gpt_variant"   weight=1.0  (100% of traffic normally)
  +-- Variant "groq_variant"  weight=0.0  (can be bumped up for A/B)
```

This is **experiment-level routing** -- completely different implementations.  
Use this when: A/B testing, canary deploys, cost experiments. (Module 3)

---

### Key Distinction

| | Provider Routing | Variant Routing |
|--|--|--|
| Purpose | Resilience / uptime | Experimentation / optimization |
| Trigger | Provider failure | Traffic weights (probabilistic) |
| User sees | Nothing (transparent) | Different responses (intentional) |
| Config | `routing = [...]` in model | `weight =` in variant |
| Module | This module | Module 3 |

## 1.3 -- A Note on Azure vs Groq

In many production setups, the fallback pattern is **OpenAI --> Azure OpenAI**:  
same GPT model, different hosting, same output quality.

Since we are using OpenAI + Groq in this course (no Azure account needed),  
we will demonstrate: **OpenAI --> Groq**.

The TOML config is **identical in structure** -- only the provider type and model name change.  
Everything you learn here applies directly to OpenAI --> Azure, OpenAI --> Bedrock, etc.

```toml
# OpenAI -> Azure pattern (production)      # OpenAI -> Groq pattern (this course)
[models.gpt.providers.azure]                [models.gpt.providers.groq]
type = "azure"                              type = "groq"
deployment_id = "gpt-4o-mini"              model_name = "llama-3.1-8b-instant"
endpoint = "https://..."
```

---

---
# Part 2 -- Provider-Level Routing: OpenAI -> Groq Fallback

---

## 2.1 -- Updating tensorzero.toml

We will add two new models to `tensorzero.toml`:

**Model 1: `resilient_gpt`** -- OpenAI primary, Groq fallback (normal use)
```toml
[models.resilient_gpt]
routing = ["openai_primary", "groq_fallback"]
```

**Model 2: `demo_failover`** -- Broken provider primary, Groq rescue (failure demo)  
The broken provider points to `http://localhost:9999` -- nothing is listening there.  
This guarantees a connection failure, forcing fallback to Groq.

```toml
[models.demo_failover]
routing = ["broken_provider", "groq_rescue"]

[models.demo_failover.providers.broken_provider]
type = "openai"
model_name = "gpt-4o-mini"
api_base = "http://localhost:9999/v1"   # <-- nothing here, guaranteed failure
```

When a request hits `demo_failover`:
1. Gateway tries `broken_provider` --> connection refused (instant)
2. Gateway logs the failure internally
3. Gateway tries `groq_rescue` --> succeeds
4. Response returned to user -- they see no error

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1: Setup -- point to our existing project from Module 1
# ─────────────────────────────────────────────────────────────────
from pathlib import Path
import os

project_dir = Path("tensorzero-demo")
config_dir  = project_dir / "config"

# Verify Module 1 stack exists
assert (project_dir / "docker-compose.yml").exists(), "Run Module 1 first!"
assert (config_dir  / "tensorzero.toml").exists(),    "Run Module 1 first!"

print("Module 1 project found:")
for f in sorted(config_dir.iterdir()):
    print(f"  config/{f.name}")

GATEWAY_URL = "http://localhost:3000"

Module 1 project found:
  config/summarize_system.minijinja
  config/tensorzero.toml


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2: Write updated tensorzero.toml
# Adds multi-provider routing on top of Module 1 config.
# ─────────────────────────────────────────────────────────────────

tensorzero_toml = """\
# ================================================================
# tensorzero.toml -- Module 2: Multi-Provider Routing
# ================================================================


# ================================================================
# MODULE 1 FUNCTIONS (kept from previous module)
# ================================================================

[functions.summarize]
type = "chat"

[functions.summarize.variants.gpt_variant]
type            = "chat_completion"
model           = "openai::gpt-4o-mini"
system_template = "summarize_system.minijinja"
weight          = 1

[functions.summarize.variants.groq_variant]
type            = "chat_completion"
model           = "groq::llama-3.1-8b-instant"
system_template = "summarize_system.minijinja"
weight          = 0

[functions.chat]
type = "chat"

[functions.chat.variants.gpt_mini]
type   = "chat_completion"
model  = "openai::gpt-4o-mini"
weight = 1

[functions.chat.variants.groq_llama]
type   = "chat_completion"
model  = "groq::llama-3.3-70b-versatile"
weight = 0


# ================================================================
# MODULE 2: MULTI-PROVIDER ROUTING
# ================================================================

# ── Model 1: resilient_gpt ──────────────────────────────────────
# OpenAI as primary, Groq as fallback.
# routing = [...] defines the ORDER providers are tried.
# If openai_primary fails for any reason (outage, rate limit,
# timeout), TensorZero AUTOMATICALLY tries groq_fallback next.
# Your application code never knows a fallback happened.

[models.resilient_gpt]
routing = ["openai_primary", "groq_fallback"]

[models.resilient_gpt.providers.openai_primary]
type       = "openai"
model_name = "gpt-4o-mini"

[models.resilient_gpt.providers.groq_fallback]
type       = "groq"
model_name = "llama-3.1-8b-instant"


# ── Model 2: demo_failover ──────────────────────────────────────
# Used ONLY to demonstrate fallback behavior in this notebook.
# broken_provider points to localhost:9999 -- nothing is there.
# This causes an immediate connection failure, forcing fallback
# to groq_rescue. In real life this simulates OpenAI being down.

[models.demo_failover]
routing = ["broken_provider", "groq_rescue"]

[models.demo_failover.providers.broken_provider]
type       = "openai"
model_name = "gpt-4o-mini"
api_base   = "http://localhost:9999/v1"

[models.demo_failover.providers.groq_rescue]
type       = "groq"
model_name = "llama-3.1-8b-instant"


# ── Function: routed_chat ───────────────────────────────────────
# Normal function using the resilient_gpt model.
# Under normal conditions: served by OpenAI.
# If OpenAI is down: transparently served by Groq.

[functions.routed_chat]
type = "chat"

[functions.routed_chat.variants.resilient_variant]
type   = "chat_completion"
model  = "resilient_gpt"
weight = 1


# ── Function: failover_demo ─────────────────────────────────────
# Demo function for observing fallback behavior.
# Primary provider is intentionally broken.

[functions.failover_demo]
type = "chat"

[functions.failover_demo.variants.with_fallback]
type   = "chat_completion"
model  = "demo_failover"
weight = 1
"""

toml_file = config_dir / "tensorzero.toml"
toml_file.write_text(tensorzero_toml, encoding="utf-8")

print(f"Written: {toml_file.resolve()}")
print()
print("New models added:")
print("  resilient_gpt  -> openai_primary -> groq_fallback")
print("  demo_failover  -> broken_provider (port 9999) -> groq_rescue")
print()
print("New functions added:")
print("  routed_chat    -> resilient_variant (uses resilient_gpt)")
print("  failover_demo  -> with_fallback (uses demo_failover)")

Written: D:\UDEMY AI SECURITY\TENSORZERO LLM Gateways\tensorzero-demo\config\tensorzero.toml

New models added:
  resilient_gpt  -> openai_primary -> groq_fallback
  demo_failover  -> broken_provider (port 9999) -> groq_rescue

New functions added:
  routed_chat    -> resilient_variant (uses resilient_gpt)
  failover_demo  -> with_fallback (uses demo_failover)


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3: Restart gateway to pick up new config
# Only the gateway needs restart -- postgres and ui stay running.
# ─────────────────────────────────────────────────────────────────
import subprocess
import time
import urllib.request

print("Restarting gateway...")
result = subprocess.run(
    ["docker", "compose", "restart", "gateway"],
    cwd=str(project_dir.resolve()),
    capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())

# Wait for gateway to be healthy again
print("Waiting for gateway to be healthy...")
for attempt in range(20):
    try:
        urllib.request.urlopen("http://localhost:3000/health", timeout=3)
        print(f"Gateway healthy after {attempt + 1} attempts")
        break
    except Exception:
        time.sleep(2)
else:
    print("Gateway did not come back healthy. Check logs.")

Restarting gateway...
Container tensorzero-demo-gateway-1 Restarting 
 Container tensorzero-demo-gateway-1 Started
Waiting for gateway to be healthy...
Gateway healthy after 2 attempts


## 2.2 -- Demo: Normal Routing (OpenAI Handles Request)

When everything is working, `routed_chat` routes to `openai_primary` (first in the list).  
We can confirm this by checking which provider served the request in the gateway logs  
and in the TensorZero UI at `http://localhost:4000`.

The key field to watch: **`variant_name`** tells you which variant handled the call.  
For provider-level routing within a variant, you inspect the gateway logs.

In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4: Normal Routing -- OpenAI handles the request
# ─────────────────────────────────────────────────────────────────
from tensorzero import TensorZeroGateway
import time

QUESTION = "What are the main benefits of using an LLM gateway in production?"

print("Calling 'routed_chat' function (OpenAI primary, Groq fallback)")
print("Under normal conditions, OpenAI should handle this request.")
print()

start = time.time()

with TensorZeroGateway.build_http(gateway_url=GATEWAY_URL) as client:
    result = client.inference(
        function_name="routed_chat",
        input={"messages": [{"role": "user", "content": QUESTION}]}
    )

elapsed = time.time() - start

print(f"inference_id:  {result.inference_id}")
print(f"variant_name:  {result.variant_name}")
print(f"latency:       {elapsed:.2f}s")
print()
print("Response:")
print(result.content[0].text[:400], "...")
print()
print("Check http://localhost:4000 -> Inferences to see which provider was used.")

Calling 'routed_chat' function (OpenAI primary, Groq fallback)
Under normal conditions, OpenAI should handle this request.

inference_id:  019e97c9-36dd-7d53-a1ea-d9648032620f
variant_name:  resilient_variant
latency:       8.32s

Response:
Using a Large Language Model (LLM) gateway in production offers several key benefits:

1. **Scalability**: LLM gateways can handle large volumes of requests simultaneously, allowing for the processing of numerous queries at scale. This is crucial for applications with high user demand.

2. **Performance Optimization**: LLM gateways often include mechanisms for optimizing model performance, such as ...

Check http://localhost:4000 -> Inferences to see which provider was used.


## 2.3 -- Demo: Simulating Provider Failure and Watching Fallback

Now we call `failover_demo` -- a function whose primary provider is deliberately broken.  
The broken provider points to `http://localhost:9999` where nothing is listening.

**What will happen step by step:**

```
1. Your code calls failover_demo
2. Gateway tries broken_provider (localhost:9999)
   --> Connection refused immediately (no server there)
   --> Gateway logs this failure internally
3. Gateway tries groq_rescue (Groq API)
   --> Success: 200 OK
4. Gateway returns response to your code
   --> Your code sees a normal successful response
   --> No exception raised, no error returned
```

**The key point for course students:**  
Your application code is identical for both demos (Cells 4 and 5).  
Only the TOML configuration differs. The fallback is completely invisible to your code.

In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5: Failure Demo -- broken provider triggers Groq fallback
# Same code as Cell 4 -- only function_name changes.
# ─────────────────────────────────────────────────────────────────
from tensorzero import TensorZeroGateway
import time

print("Calling 'failover_demo' function (broken primary -> Groq rescue)")
print("Primary provider is localhost:9999 -- nothing there, will fail.")
print("Watch TensorZero silently fall back to Groq.")
print()

start = time.time()

with TensorZeroGateway.build_http(gateway_url=GATEWAY_URL) as client:
    result = client.inference(
        function_name="failover_demo",
        input={"messages": [{"role": "user", "content": QUESTION}]}
    )

elapsed = time.time() - start

print(f"inference_id:  {result.inference_id}")
print(f"variant_name:  {result.variant_name}")
print(f"latency:       {elapsed:.2f}s")
print()
print("Response (served by Groq after OpenAI 'failed'):")
print(result.content[0].text[:400], "...")
print()
print("No exception was raised -- fallback was transparent to this code.")

Calling 'failover_demo' function (broken primary -> Groq rescue)
Primary provider is localhost:9999 -- nothing there, will fail.
Watch TensorZero silently fall back to Groq.

inference_id:  019e97e2-2811-7ac2-97fb-921309f8c5a9
variant_name:  with_fallback
latency:       1.33s

Response (served by Groq after OpenAI 'failed'):
A Large Language Model (LLM) gateway can provide several benefits in production. Here are some of the main advantages:

1. **Scalability**: An LLM gateway can scale to handle large volumes of requests, making it suitable for production environments where high performance and reliability are crucial.

2. **Security**: By acting as a proxy between clients and your model, an LLM gateway can help prot ...

No exception was raised -- fallback was transparent to this code.


---
# Part 3 -- Retries: Provider-Level vs Variant-Level

---

## 3.1 -- How TensorZero Handles Retries

There are two distinct concepts that people often confuse:

### 1. Provider-Level Routing (What We Just Saw)

```
routing = ["openai", "groq"]
```

- If `openai` fails, try `groq`
- These are **different providers** (different API endpoints, different models)
- Called: **fallback routing**
- Handles: provider outages, auth failures, rate limits (429s)

### 2. Retrying the Same Provider

Sometimes a request fails transiently -- a brief network hiccup, a momentary server overload.  
In this case you want to **retry the same provider**, not switch to a different one.

TensorZero handles transient errors automatically before moving to the next provider  
in the routing list. The exact retry behavior is built into the gateway's HTTP client.

### Decision Tree for Failures:

```
Request fails
    |
    +--> Transient (5xx, timeout)?   --> Retry same provider (automatic)
    |
    +--> Persistent (401, 429, down) --> Move to next in routing list
    |
    +--> All providers exhausted?    --> Return error to caller
```

### The Mental Model: routing = your fallback chain

```toml
routing = ["primary", "secondary", "tertiary"]
```

Think of this as: *"try primary, then secondary, then tertiary"*.  
Each entry is a completely independent provider with its own credentials and endpoint.  
This gives you **N-tier redundancy** with zero code changes.

## 3.2 -- Provider-Level vs Variant-Level Fallback: The Key Difference

```
PROVIDER-LEVEL (within one model):

  Model: "resilient_gpt"         <-- same logical model
    Provider 1: OpenAI gpt-4o-mini   <-- same quality, different host
    Provider 2: Groq llama-3.1-8b   <-- different quality, different host

  When to use: You want the same (or similar) model,
               just from a different provider if the first is down.

VARIANT-LEVEL (within one function):

  Function: "summarize"           <-- same task
    Variant A: gpt-4o-mini         <-- expensive, high quality
    Variant B: llama-3.1-8b        <-- cheap, good enough

  When to use: A/B testing, cost experiments, canary deploys.
               Traffic is SPLIT between variants, not sequentially tried.
```

The most important distinction:
- **Provider routing** = sequential (try A, if fails try B)
- **Variant routing** = probabilistic (70% go to A, 30% go to B)

In production you often combine both:
```
Function: chat
  Variant A (70%): Model resilient_gpt --> OpenAI primary, Groq fallback
  Variant B (30%): Model groq_only     --> Groq only (testing cost savings)
```

In [7]:
# ─────────────────────────────────────────────────────────────────
# CELL 7: Compare -- Single Provider vs Multi-Provider
# Make 3 requests to each function and compare reliability.
# ─────────────────────────────────────────────────────────────────
from tensorzero import TensorZeroGateway
import time

PROMPTS = [
    "What is an LLM gateway?",
    "Name one benefit of provider fallback.",
    "What does TensorZero store in PostgreSQL?",
]

print("Sending 3 requests to each function")
print()

with TensorZeroGateway.build_http(gateway_url=GATEWAY_URL) as client:

    # Single provider: direct OpenAI (from Module 1)
    print("--- chat (single provider: OpenAI only) ---")
    for i, prompt in enumerate(PROMPTS, 1):
        t = time.time()
        r = client.inference(
            function_name="chat",
            input={"messages": [{"role": "user", "content": prompt}]}
        )
        print(f"  Request {i}: OK in {time.time()-t:.2f}s -- {r.content[0].text[:60]}...")

    print()

    # Multi-provider: OpenAI primary, Groq fallback
    print("--- routed_chat (OpenAI primary, Groq fallback) ---")
    for i, prompt in enumerate(PROMPTS, 1):
        t = time.time()
        r = client.inference(
            function_name="routed_chat",
            input={"messages": [{"role": "user", "content": prompt}]}
        )
        print(f"  Request {i}: OK in {time.time()-t:.2f}s -- {r.content[0].text[:60]}...")

print()
print("Both work identically under normal conditions.")
print("The difference only shows when OpenAI has a problem.")

Sending 3 requests to each function

--- chat (single provider: OpenAI only) ---
  Request 1: OK in 5.68s -- An LLM (Large Language Model) gateway typically refers to an...
  Request 2: OK in 1.94s -- One benefit of provider fallback is increased reliability in...
  Request 3: OK in 5.65s -- TensorZero is a service that focuses on enabling the storage...

--- routed_chat (OpenAI primary, Groq fallback) ---
  Request 1: OK in 5.53s -- An LLM gateway typically refers to a system or interface tha...
  Request 2: OK in 1.89s -- One benefit of provider fallback is increased reliability. I...
  Request 3: OK in 4.08s -- As of my last knowledge update in October 2023, TensorZero i...

Both work identically under normal conditions.
The difference only shows when OpenAI has a problem.


## The Real Lesson: Reliability, Not the Output

This lesson is not about what you **see**. It's about what you **don't see**.

### With `chat` (Single Provider)

All 3 requests worked ✅

But that happened because OpenAI was available at the time.

If OpenAI experiences an outage:

- Request 1 ❌ Fails
- Request 2 ❌ Fails
- Request 3 ❌ Fails

Your application has no backup provider, so every request depends entirely on OpenAI.

### With `routed_chat` (Multi-Provider)

All 3 requests worked ✅

However, the important difference is that you now have **failover protection**.

OpenAI is still your primary provider, but Groq is standing by as a backup.

If OpenAI experiences an outage:

- Request 1 ✅ Automatically routed to Groq
- Request 2 ✅ Automatically routed to Groq
- Request 3 ✅ Automatically routed to Groq

From the user's perspective, the requests continue to succeed.

In [8]:
## FAILURE SECANRIO WHEN OPENAI is Down midway


import subprocess, time, urllib.request

extra_config = """
[models.broken_only]
routing = ["dead_provider"]

[models.broken_only.providers.dead_provider]
type       = "openai"
model_name = "gpt-4o-mini"
api_base   = "http://localhost:9999/v1"

[functions.chat_no_fallback]
type = "chat"

[functions.chat_no_fallback.variants.single_broken]
type   = "chat_completion"
model  = "broken_only"
weight = 1
"""

existing = (config_dir / "tensorzero.toml").read_text(encoding="utf-8")
(config_dir / "tensorzero.toml").write_text(existing + extra_config, encoding="utf-8")

subprocess.run(["docker", "compose", "restart", "gateway"],
               cwd=str(project_dir.resolve()), capture_output=True)

for _ in range(15):
    try:
        urllib.request.urlopen("http://localhost:3000/health", timeout=2)
        print("Gateway ready.")
        break
    except Exception:
        time.sleep(1)


Gateway ready.


In [11]:
from tensorzero import TensorZeroGateway
import time

print("Simulating: OpenAI goes DOWN mid-session")
print()

with TensorZeroGateway.build_http(gateway_url=GATEWAY_URL) as client:

    print("--- chat_no_fallback (no safety net) ---")

    # Request 1 -- use real chat function (OpenAI UP)
    t = time.time()
    r = client.inference(
        function_name="chat",
        input={"messages": [{"role": "user", "content": "What is an LLM gateway?"}]}
    )
    print(f"  Request 1: OK in {time.time()-t:.2f}s -- {r.content[0].text[:60]}...")

    # Requests 2 & 3 -- OpenAI "goes down" (switch to broken function)
    for i, prompt in enumerate(["Name one benefit of provider fallback.",
                                  "What does TensorZero store in PostgreSQL?"], 2):
        t = time.time()
        try:
            r = client.inference(
                function_name="chat_no_fallback",
                input={"messages": [{"role": "user", "content": prompt}]}
            )
            print(f"  Request {i}: OK in {time.time()-t:.2f}s -- {r.content[0].text[:60]}...")
        except Exception as e:
            print(f"  Request {i}: FAILED in {time.time()-t:.2f}s -- app crashes here")

    print()

    print("--- failover_demo (Groq rescue) ---")

    # Request 1 -- working (same as above)
    t = time.time()
    r = client.inference(
        function_name="chat",
        input={"messages": [{"role": "user", "content": "What is an LLM gateway?"}]}
    )
    print(f"  Request 1: OK in {time.time()-t:.2f}s -- {r.content[0].text[:60]}...")

    # Requests 2 & 3 -- OpenAI "goes down" but Groq rescues
    for i, prompt in enumerate(["Name one benefit of provider fallback.",
                                  "What does TensorZero store in PostgreSQL?"], 2):
        t = time.time()
        r = client.inference(
            function_name="failover_demo",
            input={"messages": [{"role": "user", "content": prompt}]}
        )
        print(f"  Request {i}: OK in {time.time()-t:.2f}s -- {r.content[0].text[:60]}...")

print()
print("Request 1 passed for both -- OpenAI was up.")
print("Requests 2-3: no fallback = crashed, with fallback = survived.")


Simulating: OpenAI goes DOWN mid-session

--- chat_no_fallback (no safety net) ---
  Request 1: OK in 5.56s -- An LLM (Large Language Model) gateway generally refers to an...
  Request 2: FAILED in 0.04s -- app crashes here
  Request 3: FAILED in 0.04s -- app crashes here

--- failover_demo (Groq rescue) ---
  Request 1: OK in 7.44s -- An LLM gateway typically refers to a hub or interface that f...
  Request 2: OK in 0.58s -- One benefit of provider fallback is that it allows applicati...
  Request 3: OK in 0.47s -- Unfortunately, I am not familiar with the data type `TensorZ...

Request 1 passed for both -- OpenAI was up.
Requests 2-3: no fallback = crashed, with fallback = survived.


---
# Part 4 -- Variant Fallbacks: Swapping the Whole Implementation

---

## 4.1 -- What Does "Swap the Whole Implementation" Mean?

Provider routing swaps the **host** but keeps the model family similar.  
Sometimes you need to swap the **entire approach** -- different model, different prompt, different quality tier.

Example scenario:
```
Primary:  GPT-4o (expensive, slow, very high quality)   weight=1.0
Fallback: Groq Llama-3.3-70B (cheap, fast, good quality) weight=0.0

Normally: All traffic -> GPT-4o
If budget alarm fires: bump Groq weight to 1.0, GPT-4o to 0.0
If GPT-4o is down: emergency switch
```

This is a **config change**, not a code deployment.  
Edit `tensorzero.toml`, restart gateway. Done in seconds.

### Emergency Swap Pattern:

```toml
# Normal operation:
[functions.answer.variants.expensive]
model  = "openai::gpt-4o"
weight = 1     # all traffic here

[functions.answer.variants.budget]
model  = "groq::llama-3.3-70b-versatile"
weight = 0     # standby, ready to activate

# Emergency (just change these two numbers + restart gateway):
# weight = 0   (expensive)
# weight = 1   (budget)
```

### Why This Is Powerful:
- No code change in your application
- No redeployment
- Rollback is equally instant (flip the weights back)
- Can be automated via API (TensorZero admin API -- advanced modules)

In [12]:
# ─────────────────────────────────────────────────────────────────
# CELL 8: Simulate Emergency Variant Swap
# Step 1: Write config with GPT-4o-mini as primary (normal)
# Step 2: "Emergency" -- switch to Groq by flipping weights
# Step 3: Switch back (rollback)
# ─────────────────────────────────────────────────────────────────
import subprocess
import time
import urllib.request

def write_toml_with_weights(gpt_weight: int, groq_weight: int):
    """Write tensorzero.toml with specified variant weights and restart gateway."""

    toml = f"""\
# Module 2 config -- variant swap demo

[functions.summarize]
type = "chat"

[functions.summarize.variants.gpt_variant]
type            = "chat_completion"
model           = "openai::gpt-4o-mini"
system_template = "summarize_system.minijinja"
weight          = 1

[functions.summarize.variants.groq_variant]
type            = "chat_completion"
model           = "groq::llama-3.1-8b-instant"
system_template = "summarize_system.minijinja"
weight          = 0

[functions.chat]
type = "chat"

[functions.chat.variants.gpt_mini]
type   = "chat_completion"
model  = "openai::gpt-4o-mini"
weight = 1

[functions.chat.variants.groq_llama]
type   = "chat_completion"
model  = "groq::llama-3.3-70b-versatile"
weight = 0

[models.resilient_gpt]
routing = ["openai_primary", "groq_fallback"]

[models.resilient_gpt.providers.openai_primary]
type       = "openai"
model_name = "gpt-4o-mini"

[models.resilient_gpt.providers.groq_fallback]
type       = "groq"
model_name = "llama-3.1-8b-instant"

[models.demo_failover]
routing = ["broken_provider", "groq_rescue"]

[models.demo_failover.providers.broken_provider]
type       = "openai"
model_name = "gpt-4o-mini"
api_base   = "http://localhost:9999/v1"

[models.demo_failover.providers.groq_rescue]
type       = "groq"
model_name = "llama-3.1-8b-instant"

[functions.routed_chat]
type = "chat"

[functions.routed_chat.variants.resilient_variant]
type   = "chat_completion"
model  = "resilient_gpt"
weight = 1

[functions.failover_demo]
type = "chat"

[functions.failover_demo.variants.with_fallback]
type   = "chat_completion"
model  = "demo_failover"
weight = 1

# Emergency swap function
[functions.emergency_chat]
type = "chat"

[functions.emergency_chat.variants.gpt_primary]
type   = "chat_completion"
model  = "openai::gpt-4o-mini"
weight = {gpt_weight}

[functions.emergency_chat.variants.groq_standby]
type   = "chat_completion"
model  = "groq::llama-3.3-70b-versatile"
weight = {groq_weight}
"""

    (config_dir / "tensorzero.toml").write_text(toml, encoding="utf-8")

    subprocess.run(
        ["docker", "compose", "restart", "gateway"],
        cwd=str(project_dir.resolve()),
        capture_output=True
    )
    for _ in range(15):
        try:
            urllib.request.urlopen("http://localhost:3000/health", timeout=2)
            break
        except Exception:
            time.sleep(1)

print("Helper function defined: write_toml_with_weights(gpt_weight, groq_weight)")
print("We will use this to simulate an emergency variant swap.")

Helper function defined: write_toml_with_weights(gpt_weight, groq_weight)
We will use this to simulate an emergency variant swap.


In [13]:
# ─────────────────────────────────────────────────────────────────
# CELL 9: Normal State -- GPT-4o-mini handles all traffic
# ─────────────────────────────────────────────────────────────────
from tensorzero import TensorZeroGateway

print("=== NORMAL STATE: gpt_primary weight=1, groq_standby weight=0 ===")
write_toml_with_weights(gpt_weight=1, groq_weight=0)
print("Config updated. Gateway restarted.")
print()

with TensorZeroGateway.build_http(gateway_url=GATEWAY_URL) as client:
    r = client.inference(
        function_name="emergency_chat",
        input={"messages": [{"role": "user", "content": "Say 'hello' in one word."}]}
    )
    print(f"variant_name: {r.variant_name}   <-- should be 'gpt_primary'")
    print(f"response:     {r.content[0].text}")

=== NORMAL STATE: gpt_primary weight=1, groq_standby weight=0 ===
Config updated. Gateway restarted.

variant_name: gpt_primary   <-- should be 'gpt_primary'
response:     Hello!


In [14]:
# ─────────────────────────────────────────────────────────────────
# CELL 10: Emergency Swap -- flip to Groq instantly
# Scenario: GPT-4o-mini costs spiked, switch to Groq to save money.
# Zero code change. Just flip weights in config + restart gateway.
# ─────────────────────────────────────────────────────────────────
from tensorzero import TensorZeroGateway

print("=== EMERGENCY SWAP: gpt_primary weight=0, groq_standby weight=1 ===")
write_toml_with_weights(gpt_weight=0, groq_weight=1)
print("Config updated. Gateway restarted.")
print()

with TensorZeroGateway.build_http(gateway_url=GATEWAY_URL) as client:
    r = client.inference(
        function_name="emergency_chat",
        input={"messages": [{"role": "user", "content": "Say 'hello' in one word."}]}
    )
    print(f"variant_name: {r.variant_name}   <-- should be 'groq_standby'")
    print(f"response:     {r.content[0].text}")
    print()
    print("Application code is IDENTICAL to Cell 9.")
    print("Different model served -- pure config change.")

=== EMERGENCY SWAP: gpt_primary weight=0, groq_standby weight=1 ===
Config updated. Gateway restarted.

variant_name: groq_standby   <-- should be 'groq_standby'
response:     Hello

Application code is IDENTICAL to Cell 9.
Different model served -- pure config change.


In [15]:
# ─────────────────────────────────────────────────────────────────
# CELL 11: Rollback -- restore GPT-4o-mini as primary
# ─────────────────────────────────────────────────────────────────
from tensorzero import TensorZeroGateway

print("=== ROLLBACK: restoring gpt_primary weight=1, groq_standby weight=0 ===")
write_toml_with_weights(gpt_weight=1, groq_weight=0)
print("Rolled back. Gateway restarted.")
print()

with TensorZeroGateway.build_http(gateway_url=GATEWAY_URL) as client:
    r = client.inference(
        function_name="emergency_chat",
        input={"messages": [{"role": "user", "content": "Say 'hello' in one word."}]}
    )
    print(f"variant_name: {r.variant_name}   <-- back to 'gpt_primary'")
    print(f"response:     {r.content[0].text}")

=== ROLLBACK: restoring gpt_primary weight=1, groq_standby weight=0 ===
Rolled back. Gateway restarted.

variant_name: gpt_primary   <-- back to 'gpt_primary'
response:     Hello!


---
# Part 5 -- Benchmarking: Gateway Latency Overhead

---

## 5.1 -- Why Latency Overhead Matters

A common objection to using an LLM gateway: *"doesn't it add latency?"*

The honest answer: **yes, but it is negligible** compared to LLM inference time.

### What Adds Latency in an LLM Call:

```
Total latency = Network to provider + LLM inference time + Network back
                     ~20ms               500ms - 10s            ~20ms

With gateway:
Total latency = Network to gateway + Gateway processing + Network to provider + LLM inference + Network back
                     ~0.1ms (localhost)    <1ms (Rust)          ~20ms              500ms-10s        ~20ms
```

The gateway adds **<1ms** because:
1. It runs **locally** (localhost, not over the internet)
2. It is written in **Rust** -- a systems language with no garbage collector, no runtime overhead

### Why Rust?

Most Python web servers (Flask, FastAPI) add 2-10ms of overhead per request due to:
- Python's Global Interpreter Lock (GIL)
- Garbage collection pauses
- Dynamic typing overhead

Rust adds essentially **zero** overhead because:
- Compiled to native machine code (no interpreter)
- Memory managed at compile time (no GC pauses)
- Zero-cost abstractions -- high-level code compiles to as fast as hand-written C
- Async I/O via Tokio (handles thousands of concurrent requests with minimal CPU)

TensorZero's p99 latency overhead is documented as **<1ms**.  
Your LLM call is 500ms-10s. The gateway overhead is literally undetectable.

In [16]:
# ─────────────────────────────────────────────────────────────────
# CELL 12: Benchmark Setup
# We will make N requests both ways and compare per-request latency.
# Using a short prompt to minimize LLM inference time variance.
# N=10 to keep cost and time reasonable.
# ─────────────────────────────────────────────────────────────────
import time
import statistics
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(project_dir / ".env")

N = 10   # number of requests per method (increase for more accurate results)
BENCH_PROMPT = "Reply with exactly one word: yes"

# Direct OpenAI client
direct_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# TensorZero client (OpenAI SDK pointed at gateway)
tz_client = OpenAI(
    api_key="not-used",
    base_url="http://localhost:3000/openai/v1"
)

print(f"Benchmark setup complete.")
print(f"Will run {N} requests per method.")
print(f"Prompt: '{BENCH_PROMPT}'")

Benchmark setup complete.
Will run 10 requests per method.
Prompt: 'Reply with exactly one word: yes'


In [23]:
# ─────────────────────────────────────────────────────────────────
# CELL 13: Benchmark -- Direct OpenAI calls
# ─────────────────────────────────────────────────────────────────

print(f"Running {N} direct OpenAI requests...")
direct_times = []

for i in range(N):
    start = time.perf_counter()
    response = direct_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Be concise."},
            {"role": "user",   "content": BENCH_PROMPT}
        ],
        max_tokens=5
    )
    elapsed = (time.perf_counter() - start) * 1000  # ms
    direct_times.append(elapsed)
    print(f"  [{i+1:2d}/{N}] {elapsed:6.0f}ms  -- '{response.choices[0].message.content.strip()}'")

print()
print(f"Direct OpenAI stats ({N} requests):")
print(f"  Mean:   {statistics.mean(direct_times):.0f}ms")
print(f"  Median: {statistics.median(direct_times):.0f}ms")
print(f"  StdDev: {statistics.stdev(direct_times):.0f}ms")
print(f"  Min:    {min(direct_times):.0f}ms")
print(f"  Max:    {max(direct_times):.0f}ms")

Running 10 direct OpenAI requests...
  [ 1/10]    817ms  -- 'Yes.'
  [ 2/10]   1052ms  -- 'Yes.'
  [ 3/10]    803ms  -- 'Yes.'
  [ 4/10]    715ms  -- 'Yes'
  [ 5/10]    977ms  -- 'Yes.'
  [ 6/10]    758ms  -- 'Yes.'
  [ 7/10]   1603ms  -- 'Yes'
  [ 8/10]   1196ms  -- 'Yes.'
  [ 9/10]    769ms  -- 'Yes.'
  [10/10]    948ms  -- 'Yes.'

Direct OpenAI stats (10 requests):
  Mean:   964ms
  Median: 883ms
  StdDev: 271ms
  Min:    715ms
  Max:    1603ms


In [22]:
# ─────────────────────────────────────────────────────────────────
# CELL 14: Benchmark -- Through TensorZero Gateway
# ─────────────────────────────────────────────────────────────────

print(f"Running {N} requests through TensorZero gateway...")
tz_times = []

for i in range(N):
    start = time.perf_counter()
    response = tz_client.chat.completions.create(
        model="tensorzero::function_name::chat",
        messages=[{"role": "user", "content": BENCH_PROMPT}],
        max_tokens=5
    )
    elapsed = (time.perf_counter() - start) * 1000  # ms
    tz_times.append(elapsed)
    print(f"  [{i+1:2d}/{N}] {elapsed:6.0f}ms  -- '{response.choices[0].message.content.strip()}'")

print()
print(f"TensorZero gateway stats ({N} requests):")
print(f"  Mean:   {statistics.mean(tz_times):.0f}ms")
print(f"  Median: {statistics.median(tz_times):.0f}ms")
print(f"  StdDev: {statistics.stdev(tz_times):.0f}ms")
print(f"  Min:    {min(tz_times):.0f}ms")
print(f"  Max:    {max(tz_times):.0f}ms")

Running 10 requests through TensorZero gateway...
  [ 1/10]   1040ms  -- 'Yes'
  [ 2/10]    883ms  -- 'Yes.'
  [ 3/10]    708ms  -- 'Yes.'
  [ 4/10]    768ms  -- 'Yes.'
  [ 5/10]    820ms  -- 'Yes'
  [ 6/10]    807ms  -- 'Yes'
  [ 7/10]    672ms  -- 'Yes'
  [ 8/10]   1580ms  -- 'Yes'
  [ 9/10]    744ms  -- 'Yes.'
  [10/10]    744ms  -- 'Yes.'

TensorZero gateway stats (10 requests):
  Mean:   877ms
  Median: 788ms
  StdDev: 268ms
  Min:    672ms
  Max:    1580ms


In [19]:
# ─────────────────────────────────────────────────────────────────
# CELL 15: Benchmark Results -- Side by Side
# ─────────────────────────────────────────────────────────────────

direct_mean = statistics.mean(direct_times)
tz_mean     = statistics.mean(tz_times)
overhead_ms = tz_mean - direct_mean
overhead_pct = (overhead_ms / direct_mean) * 100

print("=" * 55)
print("  BENCHMARK RESULTS")
print("=" * 55)
print(f"  {'Method':<30} {'Mean':>8} {'Median':>8}")
print(f"  {'-'*46}")
print(f"  {'Direct OpenAI':<30} {direct_mean:>7.0f}ms {statistics.median(direct_times):>7.0f}ms")
print(f"  {'Via TensorZero Gateway':<30} {tz_mean:>7.0f}ms {statistics.median(tz_times):>7.0f}ms")
print(f"  {'-'*46}")
print(f"  Gateway overhead:              {overhead_ms:>+7.0f}ms ({overhead_pct:+.1f}%)")
print("=" * 55)
print()

if abs(overhead_ms) < 50:
    print("Overhead is within normal network variance.")
    print("The gateway adds negligible latency (<1ms of actual processing).")
    print("The variance you see is network jitter to OpenAI's servers.")
else:
    print(f"Overhead: {overhead_ms:.0f}ms")
    print("Note: LLM response times are highly variable.")
    print("Run with larger N (50-100) for more stable comparison.")

print()
print("Key insight: The ~500ms-2s you see is OpenAI's inference time.")
print("TensorZero's actual processing overhead is <1ms (Rust, localhost).")

  BENCHMARK RESULTS
  Method                             Mean   Median
  ----------------------------------------------
  Direct OpenAI                      925ms     823ms
  Via TensorZero Gateway            1009ms     750ms
  ----------------------------------------------
  Gateway overhead:                  +84ms (+9.1%)

Overhead: 84ms
Note: LLM response times are highly variable.
Run with larger N (50-100) for more stable comparison.

Key insight: The ~500ms-2s you see is OpenAI's inference time.
TensorZero's actual processing overhead is <1ms (Rust, localhost).
